#### Code for trying differnt experiments for augmenting data.



Train the model previously and extract genrated samples from feature map when training the model. 
how do i do this? 




In [2]:
from torch.utils.data import DataLoader, Dataset
import nibabel as nib
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import random
from config import Database_config, Training_config
from dataloader import get_dataloader
import copy
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
import seaborn as sns



/home/magd6292/miniconda3/envs/multiunet2/lib/python3.10/site-packages/monai/utils/tf32.py:66: UserWarning: torch.backends.cuda.matmul.allow_tf32 = True by default.
  This value defaults to True when PyTorch version in [1.7, 1.11] and may affect precision.
  See https://docs.monai.io/en/latest/precision_accelerating.html#precision-and-accelerating
  warnings.warn(


In [7]:
train_config = Training_config
database_config = Database_config
datasetlist = ["BRATS","WMH"]
cropped_input_size = [128, 128, 128]
k_fold = None
channels_copy = copy.deepcopy(database_config.channels)

# data_size
train_size = database_config.train_size
data_size = 10
# data_size = min(train_size[dataset] for dataset in datasetlist)

# load data

train_loaders, val_loader, data_loader_map = get_dataloader(
    train_config,
    database_config,
    datasetlist,
    cropped_input_size,
    data_size,
    channels_copy,
    k_fold,
)

# just put all the training data from every modaility into a single list 
def collect_train_data(train_loaders):
    train_data = []
    for dataset in datasetlist:
        for batch_data in zip(*train_loaders):
            loader_index = data_loader_map[dataset]
            batch_data = batch_data[loader_index]
            img, label = batch_data
            train_data.append((img, label))
    
    shuffle_data = random.sample(train_data, len(train_data))
    print("total_train_data_len:", len(train_data))
    return shuffle_data


def separate_into_single_modality(train_data):
    separated_data = []
    
    for img, label in train_data:
        for idx in range (img.shape[0]):
           for i in range (img.shape[1]):
                if i < img.shape[1]:  # Ensure the index is within bounds
                    separated_data.append((torch.unsqueeze(img[idx, i, :, :, :],0), label[idx, :, :, :, :]))
                
    return separated_data



train_data = collect_train_data(train_loaders)

separated_data = separate_into_single_modality(train_data)


print("total_separated_data_len:", len(separated_data))
#############################################


# returns a list of data with label for each individual modality. 



Training:  BRATS
Training:  WMH


total_train_data_len: 10
total_separated_data_len: 60


In [8]:
# model components

# Define the encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, latent_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3_mean = nn.Linear(hidden_dim2, latent_dim)
        self.fc3_log_var = nn.Linear(hidden_dim2, latent_dim)

    def forward(self, x) -> torch.tensor:
        h = torch.relu(self.fc1(x))
        h = torch.relu(self.fc2(h))
        z_mean = self.fc3_mean(h)           # calculate mean 
        z_log_var = self.fc3_log_var(h)     # calculate log variance
        return z_mean, z_log_var


# Sampling function
def sampling(z_mean, z_log_var):
    std = torch.exp(0.5 * z_log_var)
    eps = torch.randn_like(std)
    return z_mean + eps * std


# Define the decoder
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim2, hidden_dim1, output_dim):
        super(Decoder, self).__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim2)
        self.fc2 = nn.Linear(hidden_dim2, hidden_dim1)
        self.fc3 = nn.Linear(hidden_dim1, output_dim)

    def forward(self, z):
        h = torch.relu(self.fc1(z))
        h = torch.relu(self.fc2(h))
        x_recon = torch.sigmoid(self.fc3(h))
        return x_recon


# Define the VAE model
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, latent_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim1, hidden_dim2, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim2, hidden_dim1, input_dim)

    def forward(self, x):
        z_mean, z_log_var = self.encoder(x)
        z = sampling(z_mean, z_log_var)
        x_recon = self.decoder(z)
        return x_recon, z_mean, z_log_var



In [5]:
# Instantiate the VAE model

hidden_dim1 = 64
hidden_dim2 = 32
latent_dim = 2
input_dim = 1
vae = VAE(input_dim, hidden_dim1, hidden_dim2, latent_dim)


# Define the loss function
def vae_loss(recon_x, x, z_mean, z_log_var):
    recon_loss = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    kl_divergence = -0.5 * torch.sum(1 + z_log_var - z_mean.pow(2) - z_log_var.exp())
    return recon_loss + kl_divergence


optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

# Training function
def train_vae(train_loaders, num_epochs=10):

   
    for epoch in range(num_epochs):
       
        train_loss = 0
        outputs = []
       
        for batch_data in tqdm(zip(*train_loaders), desc=f"Training VAE for  epoch {epoch}"):

            for data in datasetlist:
                loader_index = data_loader_map[data]
                batch_data = batch_data[loader_index]
                batch_data = batch_data.view(-1, batch_data.size(0))  # Flatten the image tensor
                input_dim = batch_data.shape[1]
                vae = VAE(input_dim, hidden_dim1, hidden_dim2, latent_dim)
                recon_x, z_mean, z_log_var = vae(batch_data)
                loss = vae_loss(recon_x, batch_data, z_mean, z_log_var)
                outputs.append(recon_x, z_mean, z_log_var)
                train_loss += loss.item()
            

            optimizer.zero_grad()
            loss = vae_loss(recon_x, batch_data, z_mean, z_log_var)
            loss.backward()
            optimizer.step()
                # 
        print(f"Epoch {epoch + 1}, Loss: {train_loss / len(train_loaders)}")

# Train the VAE
train_vae(train_loaders)

Training VAE for  epoch 0: 0it [00:01, ?it/s]


AttributeError: 'list' object has no attribute 'view'

In [ ]:
# generate a new example using posterior sampling. 

#load up the saved model

model = VAE(input_dim, hidden_dim1, hidden_dim2, latent_dim)

def generate_sample(model,dataloader,n_samples=1):
    model.eval()
    
    with torch.no_grad():
        for data in dataloader:
            img = data[0]
            img = img.view(-1, img.size(0))  # Flatten the image tensor
            input_dim = img.shape[1]

            _, z_mean, z_log_var = model.encoder(img)

            input_sample = sampling(z_mean, z_log_var)

            recon_imgs= vae.decoder(input_sample)




with torch.no_grad():
    z = torch.randn(1, latent_dim)
    sample = vae.decoder(z).view(1, 1, *cropped_input_size).cpu().numpy()
    
    
    # Plot the generated image
    plt.imshow(sample[0, :, :, cropped_input_size[2] // 2], cmap="gray")
    plt.show()


